In [13]:
import copy

import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
from solver import solve_pdpcr,mass_solve, SolvedInstance
import visualization
from operator import itemgetter
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import time

In [ ]:
"""
SOLVE_METHOD ={
1: solve_vrp_sumofcosts,
2: solve_flow_makespan,
3: solve_vrp_makespan,
4: solve_vrp_ver2
}
"""

method = 2
random_data = False
total_robots = 5
total_tasks = 50
data_name = 'data4'

In [ ]:

is_depot_task = False
if random_data:
    raw = gen_data(total_robots, total_tasks, False, is_depot_task)
else:
    raw = get_data(data_name,'csv', is_depot_task)

In [ ]:
try:
    raw.depot.coor = raw.depot.pickup_coor
except Exception as e:
    pass
visualization.plot_map(raw)

In [ ]:
processed_data = process(raw, method) 
solve_result = solve_pdpcr(processed_data, total_robots, method)

In [ ]:
solve_result['sol']

In [ ]:
sol = solve_result['sol']
x = solve_result['x']
f = solve_result['f']

In [ ]:
sol.get_value_df(f)

### Quick visualization

In [ ]:
df_x = sol.get_value_df(x,key_column_names=['from','to'])
df_x_moved = df_x[df_x['value']!=0]
np_moved = df_x_moved.to_numpy()
task_idx_mapper = {task.index: task for task in processed_data.tasks}
task_idx_mapper[0] =processed_data.depot

In [ ]:
solve_result['z']

In [ ]:
processed_data.tasks

In [ ]:
processed_data.depot

In [ ]:
depot = raw.depot
plt.scatter(depot.coor[0], depot.coor[1],c= 'C3', label = 'depot')
for moved_edge in np_moved:
    from_task_idx = int(moved_edge[0])
    to_task_idx = int(moved_edge[1])
    if from_task_idx ==0:
        frm_task_dest = task_idx_mapper[from_task_idx].coor
    else:
        frm_task_dest = task_idx_mapper[from_task_idx].dest_coor
    if to_task_idx ==0:
        to_task_pickup = task_idx_mapper[to_task_idx].coor
    else:
        to_task_pickup = task_idx_mapper[to_task_idx].t
    xs = [frm_task_dest[0],to_task_pickup[0]]
    ys = [frm_task_dest[1],to_task_pickup[1]]
    plt.plot(xs, ys)
for task in raw.tasks:
        pickup_coor = task.pickup_coor
        dest_coor = task.dest_coor
        plt.scatter(pickup_coor[0], pickup_coor[1],c = 'C0',)
        plt.scatter(dest_coor[0], dest_coor[1], c ='C1',)
        xs = [pickup_coor[0],dest_coor[0]]
        ys = [pickup_coor[1],dest_coor[1]]
        plt.plot(xs, ys, c = cfg.PATH_COLOR[task.n])

In [ ]:
processed_data.tasks

In [ ]:
MOVED_EDGE_GETTER ={
    1:visualization.get_moved_edges,
    2:visualization.get_moved_edges_flow
}

In [ ]:
task_idx_mapper = {task.index: task for task in processed_data.tasks}

In [ ]:
node_idx_mapper = {node.index: node for node in processed_data.tasks}
moved_edges = MOVED_EDGE_GETTER[method](solve_result)
paths = visualization.get_paths(moved_edges,total_robots)
tasks_path = visualization.get_tasks_path(paths, processed_data.nodes)
paths_coor = visualization.get_paths_coor(paths,node_idx_mapper)
edge_paths = visualization.get_edge_paths(paths)
if method ==2:
    node_idx_mapper = {node.index: node for node in processed_data.nodes[:-1]}

## Visualization

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()
V =[node for node in node_idx_mapper.keys()]
E = moved_edges
pos = {idx: node.coor for idx,node in node_idx_mapper.items()}
        
G.add_nodes_from(V)        
G.add_edges_from(E)

color_map = []
for node in V:
    if node ==0:
        color_map.append('red')
    elif node_idx_mapper[node].node_type ==1:
        color_map.append('C0')
    elif node_idx_mapper[node].node_type ==2:
        color_map.append('C1')

In [ ]:
edges_color = []
edges_style = []
for edge in E:
    for i in range(len(edge_paths)):
        if edge in edge_paths[i]:
            edges_color.append(f'C{i}')
            edges_style.append((1,(3+i*2,5)))

In [ ]:
nx.draw(G,pos = pos, node_color = color_map,node_size =50, edge_color = edges_color, style = edges_style, arrowsize =15)
if random_data == False:
    plt.savefig(f"img/{data_name}result.png", format="PNG")
else:
    plt.savefig(f"img/{seed_num}_{current_time}result_random.png", format="PNG")
plt.show()

## Mass Solve

In [ ]:
Rs = [2,3,4]
Ts = [5,10,15]

Rs = [1,2] 
Ts = [5]
total_instances = 5
cfg.LOG_OUTPUT = False
full_solved_instances = []
for T in Ts:
    for R in Rs:
        print(f'__________________________solving for {T} tasks and {R} robot__________________________')
        solved_instances = mass_solve(T,R,2,total_instances = total_instances,log_output = False)
        [full_solved_instances.append(solved_instance) for solved_instance in solved_instances]

## Mass Solve Remastered

In [ ]:
Rs = [1, 2]
Ts = [5,10,15]
instance_each_case = 4
i = 0
is_depot_task = False
instances = [gen_data(R, T, False, is_depot_task) for R in Rs for T in Ts for _ in range(instance_each_case)]

In [ ]:
def instance_to_df(raw):
    depot = raw.depot
    t_index = [depot.task_idx]
    x_p = [depot.coor[0]]
    y_p = [depot.coor[1]]
    x_d = [depot.coor[0]]
    y_d = [depot.coor[1]]
    n = [0]
    process_time = [0]
    
    tasks = raw.tasks
    
    for task in tasks:
        t_index.append(task.index)
        x_p.append(task.pickup_coor[0])
        y_p.append(task.pickup_coor[1])
        x_d.append(task.dest_coor[0])
        y_d.append(task.dest_coor[1])
        n.append(task.n)
        process_time.append(task.process_time)
    
    instance_inter = {}
    instance_inter['index'] = t_index
    instance_inter['x_pickup'] = x_p
    instance_inter['y_pickup'] = y_p
    instance_inter['x_dest'] = x_d
    instance_inter['y_dest'] = y_d
    instance_inter['robots_required'] = n
    instance_inter['process_time'] = process_time 
    df_instance = pd.DataFrame(instance_inter).round(2)
    
    return df_instance

In [ ]:
i = 0
for instance in instances:
    df_instance = instance_to_df(instance)
    df_instance.to_csv(f"./data/generated{i}.csv", index = False)
    i = i+1

In [ ]:
method = 2
is_depot_task = False

solve_results = {}
data_names =[f"generated{i}" for i in range(24)]

for data_name in data_names:    
    raw = get_data(data_name,'csv', is_depot_task)
    total_robots = max([task.n for task in raw.tasks])
    print(f'__________Solving for data {data_name} {total_robots} robots and {len(raw.tasks)} tasks__________')
    processed_data = process(raw, method) 
    solve_result = solve_pdpcr(processed_data, total_robots, method)
    solve_results[data_name] = solve_result

In [ ]:
solve_results['generated1']['processed_nodes']

In [ ]:
dict_solved = {}
dict_solved['data'] = [data for data in solve_results]
dict_solved['obj_value'] = [solve_result['sol'].objective_value for _,solve_result in solve_results.items()]
dict_solved['solve_time'] = [solve_result['solve_time'] for _,solve_result in solve_results.items()]
dict_solved['processed_nodes'] = [solve_result['processed_nodes'] for _,solve_result in solve_results.items()]
dict_solved['gap'] = [solve_result['gap'] for _,solve_result in solve_results.items()]
dict_solved['robots'] = []
dict_solved['tasks'] = []
for data_name in data_names:    
    raw = get_data(data_name,'csv', is_depot_task)
    total_robots = max([task.n for task in raw.tasks])
    dict_solved['robots'].append(total_robots)
    dict_solved['tasks'].append(len(raw.tasks))
pd.DataFrame(dict_solved).to_csv('flow_based.csv')

In [ ]:
pd.DataFrame(dict_solved).round(2).to_csv('flow_based.csv')

In [2]:
method = 5
is_depot_task = False

solve_results_vrp = {}
data_names =[f"generated{i}" for i in range(24)]
for data_name in data_names:    
    raw = get_data(data_name,'csv', is_depot_task)
    total_robots = max([task.n for task in raw.tasks])
    print(f'__________Solving for data {data_name} {total_robots} robots and {len(raw.tasks)} tasks__________')
    processed_data = process(raw, method) 
    solve_result = solve_pdpcr(processed_data, total_robots, method)
    solve_results_vrp[data_name] = solve_result

__________Solving for data generated0 1 robots and 5 tasks__________
C:\Users\CTNGUY~1\AppData\Local\Temp\VRP_ver_2_makespan.lp
Start solving vrp-based model for makespan obj.function using CPLEX
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 13 rows and 13 columns.
MIP Presolve modified 20 coefficients.
Aggregator did 1 substitutions.
Reduced MIP has 42 rows, 36 columns, and 145 nonzeros.
Reduced MIP has 30 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.17 ticks)
Found incumbent of value 2852.000000 after 0.01 sec. (0.22 ticks)
Probing time = 0.00 sec. (0.04 ticks)
Cover probing fixed 0 vars, tightened 12 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 2 rows and 0 columns.
MIP Presolve modified 46 coefficients.
R

      0     0        cutoff           2007.9178     2007.9178       28    0.00%
Elapsed time = 0.03 sec. (2.63 ticks, tree = 0.01 MB, solutions = 5)

Root node processing (before b&c):
  Real time             =    0.03 sec. (2.63 ticks)
Parallel b&c, 6 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.03 sec. (2.63 ticks)
__________Solving for data generated3 1 robots and 5 tasks__________
C:\Users\CTNGUY~1\AppData\Local\Temp\VRP_ver_2_makespan.lp
Start solving vrp-based model for makespan obj.function using CPLEX
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 13 rows and 13 columns.
MIP Presolve modified 20 coefficients

*  4767+    1                         4120.6004      929.3160            77.45%
*  5436+  473                         4102.9771      929.3160            77.35%
*  7119+ 1101                         3990.3523      942.9346            76.37%
   7436  1472     1053.3832    12     3990.3523      942.9346    51491   76.37%
  14003  5744     1568.9897    17     3990.3523      942.9346   103084   76.37%
  17781  8585     1331.9805    11     3990.3523      942.9346   169607   76.37%
  21303 10700    infeasible           3990.3523      967.9904   234264   75.74%
  24829 12784     1454.3217    17     3990.3523      999.5835   301130   74.95%
  28410 14798     1242.5716    20     3990.3523     1013.8443   362608   74.59%
  32081 16632    infeasible           3990.3523     1026.0707   424717   74.29%
  35920 18739    infeasible           3990.3523     1029.3434   481675   74.20%
  50979 26214     1635.0609    14     3990.3523     1055.4767   708666   73.55%
Elapsed time = 3.56 sec. (3223.16 ticks,

Presolve time = 0.02 sec. (0.61 ticks)
Found incumbent of value 6072.000000 after 0.02 sec. (0.83 ticks)
Probing time = 0.00 sec. (0.22 ticks)
Cover probing fixed 0 vars, tightened 24 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 5 rows and 0 columns.
MIP Presolve modified 190 coefficients.
Reduced MIP has 127 rows, 121 columns, and 530 nonzeros.
Reduced MIP has 110 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.36 ticks)
Probing fixed 0 vars, tightened 7 bounds.
Probing time = 0.00 sec. (0.37 ticks)
Cover probing fixed 0 vars, tightened 10 bounds.
Clique table members: 67.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 6 threads.
Root relaxation solution time = 0.00 sec. (0.41 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0       

 716019 47076        cutoff           3864.7153     1909.1631 11014942   50.60%
 734222 45291     2186.1156     9     3864.7153     1931.0134 11290975   50.03%
*742966 44992      integral     0     3796.2415     1948.1727 11413129   48.68%
 752444 44550     2029.5597     9     3796.2415     1958.6770 11540594   48.40%
 770788 41830     2057.2282    11     3796.2415     1968.8843 11807462   48.14%
 788864 39469    infeasible           3796.2415     2006.0899 12101375   47.16%
Elapsed time = 41.80 sec. (41261.78 ticks, tree = 15.23 MB, solutions = 26)
 807381 36841     2289.4330    13     3796.2415     2045.3990 12392059   46.12%
 826040 34032     2312.3626     8     3796.2415     2078.7146 12591999   45.24%
 844653 29741    infeasible           3796.2415     2114.5740 12900024   44.30%
 864006 25321        cutoff           3796.2415     2170.4114 13167180   42.83%
 883481 17436    infeasible           3796.2415     2297.1357 13521507   39.49%
 903239 10317    infeasible           3796.2

  67560  2626        cutoff           4055.7604      998.2557   905828   75.39%
  81123  9621     1252.6707    23     4055.7604     1026.3175  1149732   74.69%
  95226 16057        cutoff           4055.7604     1060.2057  1382729   73.86%
 109917 22936     2536.0373    13     4055.7604     1083.6234  1621962   73.28%
 124719 29148     1622.9708    18     4055.7604     1114.4793  1875989   72.52%
 139836 36394     1249.9495    23     4055.7604     1132.1932  2092695   72.08%
 152708 39295     1820.5528    20     4055.7604     1160.6370  2294011   71.38%
 166627 44438     1210.4952    14     4055.7604     1173.6175  2544973   71.06%
 180410 48214     1327.9737    20     4055.7604     1191.4494  2755463   70.62%
 193591 51931        cutoff           4055.7604     1195.1021  3010162   70.53%
Elapsed time = 12.05 sec. (12658.87 ticks, tree = 15.13 MB, solutions = 17)
 207761 53440     1253.0777    18     4055.7604     1216.9277  3220914   70.00%
 220753 54841     1440.0312    14     4055.7

... (105 outputs left out)

  Wait time (average)   =    1.47 sec.
                          ------------
Total (root+branch&cut) = 7208.28 sec. (2430464.42 ticks)
__________Solving for data generated23 2 robots and 15 tasks__________
C:\Users\CTNGUY~1\AppData\Local\Temp\VRP_ver_2_makespan.lp
Start solving vrp-based model for makespan obj.function using CPLEX
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 48 rows and 57 columns.
MIP Presolve modified 63 coefficients.
Aggregator did 7 substitutions.
Reduced MIP has 509 rows, 466 columns, and 2259 nonzeros.
Reduced MIP has 450 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.06 sec. (3.02 ticks)
Found incumbent of value 6534.000000 after 0.13 sec. (3.80 ticks)
Probing time = 0.00 sec. (1.79 ticks)
Cover probing fixed 0 vars, tightened 48 b

 181537 108939    infeasible           5346.2895      900.5345  2698262   83.16%
*183195+110510                         5338.6374      900.5345            83.13%
 184835 112034     1938.5777    23     5338.6374      900.5345  2776730   83.13%
 189174 115858     1376.4176    36     5338.6374      900.5345  2864068   83.13%
 193560 119396     1578.7065    39     5338.6374      900.5345  2951291   83.13%
 197916 123424     1961.1442    23     5338.6374      900.5345  3044024   83.13%
*200407+126213                         5167.0517      900.5345            82.57%
 201114 126825     1471.8694    34     5167.0517      900.5345  3131790   82.57%
Elapsed time = 306.00 sec. (32519.09 ticks, tree = 56.69 MB, solutions = 13)
 205338 130216      959.6032    50     5167.0517      900.5345  3213356   82.57%
 209686 133900     1141.1556    40     5167.0517      900.5345  3295873   82.57%
 214207 138070     1642.9323    31     5167.0517      900.5345  3388444   82.57%
 217992 141204     4414.6961    

Elapsed time = 928.03 sec. (168911.37 ticks, tree = 507.05 MB, solutions = 19)
 834601 707400     2871.8499    20     4802.9643      900.5345 15769644   81.25%
 851669 724399     1844.7406    28     4802.9643      900.5345 16133483   81.25%
 868699 737824      931.1014    48     4802.9643      900.5345 16413432   81.25%
 886736 756389     1308.7919    26     4802.9643      900.5345 16830663   81.25%
 903425 771790     2040.6179    28     4802.9643      900.5345 17155271   81.25%
 919324 786815     1207.6606    48     4802.9643      900.5345 17518656   81.25%
 937554 801599     1354.6205    38     4802.9643      900.5345 17839776   81.25%
 953822 815830     1620.2982    27     4802.9643      900.5345 18103005   81.25%
 970648 834082      900.8743    84     4802.9643      900.5345 18508964   81.25%
 987652 848061     1241.9836    28     4802.9643      900.5345 18841535   81.25%
Elapsed time = 1092.67 sec. (207060.22 ticks, tree = 673.48 MB, solutions = 19)
 1004072 862390     1219.6178  

FileNotFoundError: [Errno 2] No such file or directory: 'data/generated24.csv'

In [22]:
method = 5
is_depot_task = False

data_name ='generated23'
raw = get_data(data_name,'csv', is_depot_task)
total_robots = max([task.n for task in raw.tasks])
print(f'__________Solving for data {data_name} {total_robots} robots and {len(raw.tasks)} tasks__________')
processed_data = process(raw, method) 
solve_result = solve_pdpcr(processed_data, total_robots, method)

__________Solving for data generated23 2 robots and 15 tasks__________
C:\Users\CTNGUY~1\AppData\Local\Temp\VRP_ver_2_makespan.lp
Start solving vrp-based model for makespan obj.function using CPLEX
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 48 rows and 57 columns.
MIP Presolve modified 63 coefficients.
Aggregator did 7 substitutions.
Reduced MIP has 509 rows, 466 columns, and 2259 nonzeros.
Reduced MIP has 450 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (3.02 ticks)
Found incumbent of value 6534.000000 after 0.02 sec. (3.80 ticks)
Probing time = 0.00 sec. (1.79 ticks)
Cover probing fixed 0 vars, tightened 48 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 8 rows and 0 columns.
MIP Presolve modified 888 coeffici

 184835 112034     1938.5777    23     5338.6374      900.5345  2776730   83.13%
 189174 115858     1376.4176    36     5338.6374      900.5345  2864068   83.13%
 193560 119396     1578.7065    39     5338.6374      900.5345  2951291   83.13%
 197916 123424     1961.1442    23     5338.6374      900.5345  3044024   83.13%
*200407+126213                         5167.0517      900.5345            82.57%
 201114 126825     1471.8694    34     5167.0517      900.5345  3131790   82.57%
Elapsed time = 31.24 sec. (32519.09 ticks, tree = 56.69 MB, solutions = 13)
 205338 130216      959.6032    50     5167.0517      900.5345  3213356   82.57%
 209686 133900     1141.1556    40     5167.0517      900.5345  3295873   82.57%
 214207 138070     1642.9323    31     5167.0517      900.5345  3388444   82.57%
 217992 141204     4414.6961    20     5167.0517      900.5345  3456640   82.57%
 222549 145928     1074.9913    47     5167.0517      900.5345  3557008   82.57%
 227067 150314     1328.5168    3

 851669 724399     1844.7406    28     4802.9643      900.5345 16133483   81.25%
 868699 737824      931.1014    48     4802.9643      900.5345 16413432   81.25%
 886736 756389     1308.7919    26     4802.9643      900.5345 16830663   81.25%
 903425 771790     2040.6179    28     4802.9643      900.5345 17155271   81.25%
 919324 786815     1207.6606    48     4802.9643      900.5345 17518656   81.25%
 937554 801599     1354.6205    38     4802.9643      900.5345 17839776   81.25%
 953822 815830     1620.2982    27     4802.9643      900.5345 18103005   81.25%
 970648 834082      900.8743    84     4802.9643      900.5345 18508964   81.25%
 987652 848061     1241.9836    28     4802.9643      900.5345 18841535   81.25%
Elapsed time = 162.36 sec. (207060.22 ticks, tree = 673.48 MB, solutions = 19)
 1004072 862390     1219.6178    34     4802.9643      900.5345 19169474   81.25%
 1020491 877898     1553.0750    48     4802.9643      900.5345 19522392   81.25%
 1038461 895138      900.559

Elapsed time = 439.02 sec. (512821.76 ticks, tree = 2604.49 MB, solutions = 20)
Nodefile size = 555.12 MB (315.81 MB after compression)
 2265640 2017071      900.7601    86     4751.0847      900.5345 44330292   81.05%
 2280919 2029318     3099.4891    21     4751.0847      900.5345 44633201   81.05%
 2296626 2044319     2767.3638    20     4751.0847      900.5345 44973807   81.05%
 2305146 2050151     1089.7071    42     4751.0847      900.5345 45076985   81.05%
 2312339 2059239     1649.8434    34     4751.0847      900.5345 45251035   81.05%
 2328237 2073112     1321.3743    38     4751.0847      900.5345 45583893   81.05%
 2344498 2088520      901.8599    85     4751.0847      900.5345 45960149   81.05%
 2358895 2102785     2286.5708    32     4751.0847      900.5345 46301603   81.05%
 2375052 2114488     1620.0744    37     4751.0847      900.5345 46567341   81.05%
 2389743 2128792      900.6171   106     4751.0847      900.5345 46870184   81.05%
Elapsed time = 476.19 sec. (550970

 3383854 3036358     1666.6446    37     4647.9728      900.5345 66235840   80.63%
 3399729 3052125     2055.7414    35     4647.9728      900.5345 66558489   80.63%
 3415709 3065107     1179.1813    35     4647.9728      900.5345 66834899   80.63%
 3432301 3078698     1205.4241    35     4647.9728      900.5345 67124872   80.63%
 3449194 3098229     3771.5407    28     4647.9728      900.5345 67550541   80.63%
 3463705 3108570      907.6644    68     4647.9728      900.5345 67771596   80.63%
 3480472 3131437     1265.2951    44     4647.9728      900.5345 68252160   80.63%
Elapsed time = 793.63 sec. (856252.82 ticks, tree = 4007.77 MB, solutions = 21)
Nodefile size = 1958.90 MB (1072.77 MB after compression)
 3497530 3140410     2002.4247    31     4647.9728      900.5345 68446120   80.63%
 3513534 3154972     1299.2640    34     4647.9728      900.5345 68750091   80.63%
 3528928 3171637     1104.2341    43     4647.9728      900.5345 69125539   80.63%
 3545928 3188129     1248.0765  

 4206955 3794093     2546.1833    29     4647.9728      900.5345 79864876   80.63%
Elapsed time = 1091.34 sec. (1171470.82 ticks, tree = 6359.89 MB, solutions = 21)
Nodefile size = 4307.05 MB (2345.48 MB after compression)
 4212328 3799823     1688.6068    33     4647.9728      900.5345 79953889   80.63%
 4222472 3807236     1671.7863    29     4647.9728      900.5345 80077796   80.63%
 4230015 3813445     1591.1679    34     4647.9728      900.5345 80177945   80.63%
 4232910 3816305      954.6009    53     4647.9728      900.5345 80222966   80.63%
 4239634 3821407      996.7195    37     4647.9728      900.5345 80302762   80.63%
 4246015 3826261     1556.0242    28     4647.9728      900.5345 80381270   80.63%
 4253314 3836066     2102.6689    30     4647.9728      900.5345 80532050   80.63%
 4257068 3839223     1902.7578    30     4647.9728      900.5345 80579852   80.63%
 4260751 3842140     1231.2956    41     4647.9728      900.5345 80623874   80.63%
 4266065 3847010     1751.2201

... (5 outputs left out)

 8176325 7437959      900.9946   100     4626.5225      900.5345 1.33e+08   80.54%
 8184681 7446112     1284.2743    39     4626.5225      900.5345 1.33e+08   80.54%
 8192655 7451127     1559.8465    31     4626.5225      900.5345 1.33e+08   80.54%
 8199362 7458701        cutoff           4626.5225      900.5345 1.33e+08   80.54%
Elapsed time = 3264.24 sec. (3120802.76 ticks, tree = 37638.95 MB, solutions = 22)
Nodefile size = 35579.97 MB (19535.23 MB after compression)
 8207342 7465297     1031.5343    52     4626.5225      900.5345 1.33e+08   80.54%
 8214678 7472749     1263.5917    32     4626.5225      900.5345 1.34e+08   80.54%
 8222384 7479879      904.6519    66     4626.5225      900.5345 1.34e+08   80.54%
 8229940 7485717     2434.5938    27     4626.5225      900.5345 1.34e+08   80.54%
 8237242 7494402     3004.3860    21     4626.5225      900.5345 1.34e+08   80.54%
 8245324 7499897     1245.8159    34     4626.5225      900.5345 1.34e+08   80.54%
 8254225 7506991     1752.4

Elapsed time = 3599.11 sec. (3426011.64 ticks, tree = 42659.83 MB, solutions = 23)
Nodefile size = 40599.49 MB (22289.07 MB after compression)
 8831819 8035168      900.6632    98     4613.5261      900.5345 1.41e+08   80.48%
 8840324 8043349     1638.3198    24     4613.5261      900.5345 1.41e+08   80.48%
 8848575 8051187     1152.4133    42     4613.5261      900.5345 1.41e+08   80.48%
 8854850 8058332     1277.8875    27     4613.5261      900.5345 1.41e+08   80.48%
 8864687 8064998     2474.7062    23     4613.5261      900.5345 1.41e+08   80.48%
 8872320 8073413     1029.5907    37     4613.5261      900.5345 1.41e+08   80.48%
 8880151 8080246     1434.1797    27     4613.5261      900.5345 1.42e+08   80.48%
 8886672 8087149     1126.1813    37     4613.5261      900.5345 1.42e+08   80.48%
 8896282 8096152     1063.0212    46     4613.5261      900.5345 1.42e+08   80.48%
 8905827 8105177     1230.8109    47     4613.5261      900.5345 1.42e+08   80.48%
Elapsed time = 3640.42 sec.

 11122062 10147259     3060.3958    18     4613.5261      900.5345 1.70e+08   80.48%
 11170769 10192530      905.5141    63     4613.5261      900.5345 1.71e+08   80.48%
 11213675 10230738      903.0196    57     4613.5261      900.5345 1.71e+08   80.48%
 11251380 10263074      900.8108    80     4613.5261      900.5345 1.72e+08   80.48%
 11290364 10303665      952.1744    58     4613.5261      900.5345 1.72e+08   80.48%
 11328454 10333404     1597.2121    30     4613.5261      900.5345 1.73e+08   80.48%
 11361305 10365683     1792.7269    26     4613.5261      900.5345 1.73e+08   80.48%
Elapsed time = 4987.45 sec. (4467456.55 ticks, tree = 59001.88 MB, solutions = 23)
Nodefile size = 56949.19 MB (31271.43 MB after compression)
 11395836 10395190     1571.6141    35     4613.5261      900.5345 1.73e+08   80.48%
 11425429 10423862      911.3574    57     4613.5261      900.5345 1.74e+08   80.48%
 11468896 10463187     1099.4672    33     4613.5261      900.5345 1.74e+08   80.48%
 115057

Starting limited solution polishing.
 13843054 12676729     1227.5746    44     4613.5261      900.5345 2.03e+08   80.48%
*13844624+12678958                         4585.4133      900.5345            80.36%
*13849401+12685475                         4501.3571      900.5345            79.99%
 13868428 12701759     3293.6309    27     4501.3571      900.5345 2.03e+08   79.99%
 13898154 12730329     2401.1322    22     4501.3571      900.5345 2.04e+08   79.99%
 13927088 12756089     1283.8424    34     4501.3571      900.5345 2.04e+08   79.99%
Elapsed time = 6587.63 sec. (5688199.40 ticks, tree = 76674.28 MB, solutions = 25)
Nodefile size = 74618.21 MB (41041.29 MB after compression)
 13952715 12777789     1185.4877    42     4501.3571      900.5352 2.04e+08   79.99%
 13977794 12801768     1918.9357    29     4501.3571      900.5355 2.05e+08   79.99%
*13992554+12814708                         4423.1725      900.5360            79.64%
 14003219 12824555     2139.3259    27     4423.1725   

In [28]:
solve_result['sol']

docplex.mp.solution.SolveSolution(obj=4355.4,values={x_0_15:1,x_0_16:1,x..

In [21]:
solve_result

{'sol': docplex.mp.solution.SolveSolution(obj=1668.03,values={f_0_5:1,f_1_2:1,f_..,
 'x': {(0, 1): docplex.mp.Var(type=B,name='x_0_1'),
  (0, 3): docplex.mp.Var(type=B,name='x_0_3'),
  (0, 5): docplex.mp.Var(type=B,name='x_0_5'),
  (0, 7): docplex.mp.Var(type=B,name='x_0_7'),
  (0, 9): docplex.mp.Var(type=B,name='x_0_9'),
  (0, 11): docplex.mp.Var(type=B,name='x_0_11'),
  (1, 2): docplex.mp.Var(type=B,name='x_1_2'),
  (2, 3): docplex.mp.Var(type=B,name='x_2_3'),
  (2, 5): docplex.mp.Var(type=B,name='x_2_5'),
  (2, 7): docplex.mp.Var(type=B,name='x_2_7'),
  (2, 9): docplex.mp.Var(type=B,name='x_2_9'),
  (2, 11): docplex.mp.Var(type=B,name='x_2_11'),
  (3, 4): docplex.mp.Var(type=B,name='x_3_4'),
  (4, 1): docplex.mp.Var(type=B,name='x_4_1'),
  (4, 5): docplex.mp.Var(type=B,name='x_4_5'),
  (4, 7): docplex.mp.Var(type=B,name='x_4_7'),
  (4, 9): docplex.mp.Var(type=B,name='x_4_9'),
  (4, 11): docplex.mp.Var(type=B,name='x_4_11'),
  (5, 6): docplex.mp.Var(type=B,name='x_5_6'),
  (6, 1): do

In [ ]:
raw = get_data(data_name,'csv', is_depot_task)
total_robots = max([task.n for task in raw.tasks])
print(f'__________Solving for data {data_name} {total_robots} robots and {len(raw.tasks)} tasks__________')
processed_data = process(raw, method) 
solve_result = solve_pdpcr(processed_data, total_robots, method)

In [7]:
dict_solved = {}
dict_solved['data'] = [data for data in solve_results_vrp]
dict_solved['obj_value'] = [solve_result['sol'].objective_value for _,solve_result in solve_results_vrp.items()]
dict_solved['solve_time'] = [solve_result['solve_time'] for _,solve_result in solve_results_vrp.items()]
dict_solved['processed_nodes'] = [solve_result['processed_nodes'] for _,solve_result in solve_results_vrp.items()]
dict_solved['gap'] = [solve_result['gap'] for _,solve_result in solve_results_vrp.items()]
dict_solved['robots'] = []
dict_solved['tasks'] = []
for data_name in data_names:    
    raw = get_data(data_name,'csv', is_depot_task)
    total_robots = max([task.n for task in raw.tasks])
    dict_solved['robots'].append(total_robots)
    dict_solved['tasks'].append(len(raw.tasks))
# pd.DataFrame(dict_solved).to_csv('flow_based.csv')

AttributeError: 'NoneType' object has no attribute 'objective_value'

In [50]:
dict_solved['gap'] = [solve_result['gap'] for _,solve_result in solve_results_vrp.items()]

In [85]:
dict_solved.pop( 'reported_node')

[0, 0, 0, 0, 50979]

In [79]:
dict_solved['robots'] = []
dict_solved['tasks'] = []
data_names =[f"generated{i}" for i in range(23)]
for data_name in data_names:    
    raw = get_data(data_name,'csv', is_depot_task)
    total_robots = max([task.n for task in raw.tasks])
    dict_solved['robots'].append(total_robots)
    dict_solved['tasks'].append(len(raw.tasks))
pd.DataFrame(dict_solved).round(2).to_csv('vrp_based.csv')

In [86]:
pd.DataFrame(dict_solved).round(2).to_csv('vrp_based.csv')

## Mass solve compare

In [ ]:
Rs = [1,2,3,4]
Ts = [5,10,15]

# Rs = [2,]
# Ts = [5]
total_instances = 4
cfg.LOG_OUTPUT = False
full_solved_instances_vrp = []
full_solved_instances_flow = []
for T in Ts:
    print(f"number of tasks {T}")
    for R in Rs:
        print(f"number of robots {R}")
        for instance in range(total_instances):
            print(f"current instance {instance}")
            raw = gen_data(R, T, False, False)
            processed_data_flow = process(raw,2)
            tik = time.time()
            solved_result_flow = solve_pdpcr(processed_data_flow, R, 2)
            tok = time.time()
            solved_time = tok - tik
            solved_flow = SolvedInstance(1,processed_data_flow,R,solved_result_flow,solved_time,instance)
            full_solved_instances_flow.append(solved_flow)
            
            processed_data_vrp = process(raw,5)
            tik = time.time()
            solved_result_vrp = solve_pdpcr(processed_data_vrp, R, 5)
            tok = time.time()
            solved_time = tok - tik
            solved_vrp = SolvedInstance(1,processed_data_vrp,R,solved_result_vrp,solved_time,instance)
            full_solved_instances_vrp.append(solved_vrp)

In [ ]:
# full_solved_instances_vrp[0].solve_result['sol'].objective_value - full_solved_instances_flow[0].solve_result['sol'].objective_value
for i in range(len(full_solved_instances_flow)):
    print(f'i = {i}')
    flow_sol = full_solved_instances_flow[i].solve_result['sol']
    vrp_sol = full_solved_instances_vrp[i].solve_result['sol']
    try:
        flow_obj = flow_sol.objective_value
        vrp_obj = vrp_sol.objective_value
    except Exception as e:
        print(e)
        pass
    print(flow_obj -vrp_obj)

In [ ]:
def results_to_df(full_solved_instances, compact = True):
    solved_dict = [solve_instance.__dict__ for solve_instance in full_solved_instances]
    df_solved = pd.DataFrame(solved_dict)
    if compact == True:
        df_solved.drop(['data','seed','instance_num'],axis =1, inplace = True)
    return df_solved

In [ ]:
df_results = results_to_df(full_solved_instances_vrp)
df_results = df_results.round(2)

In [ ]:
np.stack([xyz,xy],0)

In [ ]:
np.array([ef])

In [ ]:
raw.tasks[0].dest_coor

In [ ]:
def raw_to_df(raw):
    depot = raw.depot
    tasks = raw.tasks
    for task in tasks:
        np_array_input = np.concatenate(task.pickup_coor, task.dest_coor, task.n)

In [ ]:
total_tasks = [len(solved_instance.data.tasks) for solved_instance in full_solved_instances]
total_robots = [(solved_instance.total_robots) for solved_instance in full_solved_instances]
# total_nodes = [len(solved_instance.data.nodes) for solved_instance in full_solved_instances]
orig_tasks = []
for T in Ts:
    for R in Rs:
        for _ in range(total_instances):
            orig_tasks.append(T)

In [ ]:
raw.depot

In [ ]:
for task in raw.tasks

In [ ]:
df_results['total_tasks'] = total_tasks
# df_results['total_nodes'] = total_nodes
df_results['orig_tasks'] = orig_tasks

In [ ]:
df_results = df_results[['total_robots','orig_tasks','solve_time','gap']]

In [ ]:
df_results

In [ ]:
print(df_results.groupby(['total_robots','orig_tasks']).mean().reset_index().round(2).to_latex())

In [ ]:
df_final = df_results.groupby(['total_robots','orig_tasks']).mean().reset_index().round(2)
df_final

## Compare big M

In [ ]:
import copy

import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
from solver import solve_pdpcr,mass_solve, SolvedInstance
import visualization
from operator import itemgetter
import matplotlib.pyplot as plt

import pandas as pd
import time

In [ ]:
method = 2
random_data = True
total_robots = 3
total_tasks = 25

cfg.RELAXED = False
cfg.USE_INDICATOR = False
cfg.BENDER_STRAT = 0

In [ ]:
big_ms = [10,10**2,10**3,10**4, 10**5, 10**6,10**7]
big_ms = [10**5]
methods = [2]

In [ ]:
raw = gen_data(total_robots, total_tasks, False, False)
solve_results = []
for big_m in big_ms:
    cfg.BIG_M = big_m
    for method in methods:
        processed_data = process(raw, method)
        solve_result = solve_pdpcr(processed_data, total_robots, method)
        solve_results.append(solve_result)

In [ ]:
[solve_result['sol'].objective_value for solve_result in solve_results if solve_result['sol'] is not None]

## Visualize old

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()
V =[node for node in node_idx_mapper.keys()]
E = moved_edges
pos = {idx: node.coor for idx,node in node_idx_mapper.items()}
        
G.add_nodes_from(V)        
G.add_edges_from(E)

color_map = []
for node in V:
    if node ==0:
        color_map.append('red')
    elif node_idx_mapper[node].node_type ==1:
        color_map.append('C0')
    elif node_idx_mapper[node].node_type ==2:
        color_map.append('C1')

In [ ]:
rnd_list = {1:20,2:30}
max(rnd_list.values())

In [ ]:
rnd_list.values()